# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

I start with logistic regression because the outcome is binary and the model should be readable. I also compare it with a random forest, since the project’s baseline is a ranking-style signal and a tree ensemble can capture non-linear patterns that a simpler linear model may miss.

In [1]:
import json
import pandas as pd
import numpy as np
import sys
from pathlib import Path

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, average_precision_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'processed' / 'refresh_feature_vector.csv').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root from the current notebook path.')


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from scripts.ml_utils import (
    MODEL_NUMERIC_FEATURES,
    MODEL_CATEGORICAL_FEATURES,
    precision_at_k,
)


feature_path = ROOT / 'data' / 'processed' / 'refresh_feature_vector.csv'
baseline_path = ROOT / 'data' / 'processed' / 'baseline_refresh_queue.csv'
results_path = ROOT / 'outputs' / 'model_results.json'

frame = pd.read_csv(feature_path)
baseline_frame = pd.read_csv(baseline_path)

# The target is the same one used by the project scripts.
frame['is_declining_label'] = frame['trend_direction'].str.lower().eq('down').astype(int)

# Use a client-aware holdout, matching the repo's training script.
client_series = frame['client_id'].fillna('unknown').astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(42)
shuffled_clients = rng.permutation(unique_clients)
num_test_clients = max(1, int(round(len(shuffled_clients) * 0.2)))
test_clients = set(shuffled_clients[:num_test_clients])
test_mask = client_series.isin(test_clients).to_numpy()
train_mask = ~test_mask
train_idx = np.flatnonzero(train_mask)
test_idx = np.flatnonzero(test_mask)

# Build the same feature matrix that the training script uses.
numeric_frame = frame[MODEL_NUMERIC_FEATURES].apply(pd.to_numeric, errors='coerce').fillna(0)
categorical_frame = frame[MODEL_CATEGORICAL_FEATURES].fillna('unknown').astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=MODEL_CATEGORICAL_FEATURES, dummy_na=False, dtype=float)
feature_frame = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)

X_train = feature_frame.iloc[train_idx]
X_test = feature_frame.iloc[test_idx]
y_train = frame.iloc[train_idx]['is_declining_label']
y_test = frame.iloc[test_idx]['is_declining_label']

# Baseline scores from the repo's rule-based ranking.
baseline_lookup = baseline_frame.set_index('content_id')['baseline_refresh_score']
baseline_test_scores = frame.iloc[test_idx]['content_id'].map(baseline_lookup).fillna(0).to_numpy()


def metric_table(y_true, scores):
    preds = (np.asarray(scores) >= 0.5).astype(int)
    return {
        'accuracy': accuracy_score(y_true, preds),
        'precision': precision_score(y_true, preds, zero_division=0),
        'recall': recall_score(y_true, preds, zero_division=0),
        'f1': f1_score(y_true, preds, zero_division=0),
        'roc_auc': roc_auc_score(y_true, np.asarray(scores)),
        'average_precision': average_precision_score(y_true, np.asarray(scores)),
        'precision_at_20': precision_at_k(y_true, scores, 20),
        'precision_at_50': precision_at_k(y_true, scores, 50),
        'precision_at_100': precision_at_k(y_true, scores, 100),
    }


models = {
    'baseline': baseline_test_scores,
    'logistic_regression': Pipeline([
        ('scaler', StandardScaler()),
        ('model', LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)),
    ]),
    'random_forest': RandomForestClassifier(
        class_weight='balanced_subsample',
        max_depth=10,
        min_samples_leaf=25,
        n_estimators=200,
        n_jobs=-1,
        random_state=42,
    ),
}

results = {}
for name, model_or_scores in models.items():
    if name == 'baseline':
        results[name] = metric_table(y_test, model_or_scores)
    else:
        model = model_or_scores.fit(X_train, y_train)
        probs = model.predict_proba(X_test)[:, 1]
        results[name] = metric_table(y_test, probs)

comparison = pd.DataFrame(results).T
comparison[['accuracy','precision','recall','f1','roc_auc','average_precision','precision_at_20','precision_at_50','precision_at_100']].round(3)

,accuracy,precision,recall,f1,roc_auc,average_precision,precision_at_20,precision_at_50,precision_at_100
baseline,0.609,0.499,0.189,0.274,0.627,0.468,0.15,0.24,0.36
logistic_regression,0.661,0.566,0.567,0.566,0.700,0.522,0.35,0.40,0.44
random_forest,0.671,0.560,0.741,0.638,0.747,0.610,0.70,0.68,0.70


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

I use a client-based holdout split rather than a random row split. That is more honest for this problem because content from the same client can share patterns, and a random row split would let the model see very similar pages from the same client in train and test. The client holdout gives a more realistic estimate of out-of-client generalization.

In [4]:
# Section 3: Train + compare vs baseline

# Display the comparison table with selected metrics
print("Model Performance Comparison (Client-Holdout Test Set)")
print("=" * 80)
print(comparison)
print("\n")

# Save results to JSON for reproducibility
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved to: {results_path}")

Model Performance Comparison (Client-Holdout Test Set)
                     accuracy  precision  recall     f1  roc_auc  \
baseline                0.609      0.499   0.189  0.274    0.627   
logistic_regression     0.661      0.566   0.567  0.566    0.700   
random_forest           0.671      0.560   0.741  0.638    0.747   

                     average_precision  precision_at_20  precision_at_50  \
baseline                         0.468             0.15             0.24   
logistic_regression              0.522             0.35             0.40   
random_forest                    0.610             0.70             0.68   

                     precision_at_100  
baseline                         0.36  
logistic_regression              0.44  
random_forest                    0.70  


Results saved to: E:\flyrank-ml-internship-starter-main\flyrank-ml-internship-starter-main\outputs\model_results.json


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

The table below compares the baseline score against the two learned models on the same client-holdout test set. I focus on ranking metrics because the task is to surface the most promising decline cases first.

comparison.round(3)

In [2]:
comparison = pd.DataFrame(results).T
comparison = comparison[['accuracy','precision','recall','f1','roc_auc','average_precision','precision_at_20','precision_at_50','precision_at_100']].round(3)
comparison

,accuracy,precision,recall,f1,roc_auc,average_precision,precision_at_20,precision_at_50,precision_at_100
baseline,0.609,0.499,0.189,0.274,0.627,0.468,0.15,0.24,0.36
logistic_regression,0.661,0.566,0.567,0.566,0.700,0.522,0.35,0.40,0.44
random_forest,0.671,0.560,0.741,0.638,0.747,0.610,0.70,0.68,0.70


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

The random forest improves the ranking metrics, especially precision@50 and average precision, but it still misses many true declines. The strongest signals appear to be visibility and freshness-related features such as days with impressions, log impressions, average position, and content age. That is plausible for the task, but it is also a reminder that the model is mostly learning a 'visible and stale' pattern rather than a fully causal explanation.

In [5]:
# Error analysis: inspect the most confident false positives and false negatives.
model = models['random_forest']
probs = model.predict_proba(X_test)[:, 1]

pred_frame = pd.DataFrame({
    'client_id': frame.iloc[test_idx]['client_id'],
    'content_id': frame.iloc[test_idx]['content_id'],
    'true_label': y_test.to_numpy(),
    'probability': probs,
    'prediction': (probs >= 0.5).astype(int),
    'impressions_90d': frame.iloc[test_idx]['impressions_90d'],
    'avg_position': frame.iloc[test_idx]['avg_position'],
    'content_age_days': frame.iloc[test_idx]['content_age_days'],
})

false_positives = pred_frame[(pred_frame['prediction'] == 1) & (pred_frame['true_label'] == 0)].sort_values('probability', ascending=False).head(5)
false_negatives = pred_frame[(pred_frame['prediction'] == 0) & (pred_frame['true_label'] == 1)].sort_values('probability', ascending=False).head(5)

print('False positives:')
print(false_positives[['content_id','probability','impressions_90d','avg_position','content_age_days']].to_string(index=False))
print('\nFalse negatives:')
print(false_negatives[['content_id','probability','impressions_90d','avg_position','content_age_days']].to_string(index=False))


False positives:
          content_id  probability  impressions_90d  avg_position  content_age_days
content_d2dffcc697a4     0.737431             5091          14.1               144
content_00603b0349b4     0.735212             1076          25.6               125
content_e55b8ab078b0     0.733797              369          21.8               112
content_f5013794ba57     0.732604              881          15.7               175
content_ed3a7fd12cf8     0.732316              411          35.2               175

False negatives:
          content_id  probability  impressions_90d  avg_position  content_age_days
content_c17d3a5e3566     0.499263               17          17.2               104
content_fe358b5427df     0.499082               34          26.8               295
content_1df45ca5a295     0.499048               20          12.3               112
content_0c0eadeecab9     0.498745               25           8.5               287
content_47c38e5820a7     0.496540            22617  

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [ ]:
checks = {
    'data_loaded': isinstance(frame, pd.DataFrame) and isinstance(baseline_frame, pd.DataFrame),
    'split_created': (
        isinstance(X_train, pd.DataFrame)
        and isinstance(X_test, pd.DataFrame)
        and len(train_idx) > 0
        and len(test_idx) > 0
    ),
    'models_compared': (
        isinstance(comparison, pd.DataFrame)
        and {'baseline', 'logistic_regression', 'random_forest'}.issubset(set(comparison.index))
    ),
    'results_saved': isinstance(results_path, Path) and results_path.exists(),
}

for name, ok in checks.items():
    print(f'{name}: {"PASS" if ok else "FAIL"}')

assert all(checks.values()), 'Self-check failed: one or more required items are missing or invalid.'
print('Self-check completed. Notebook is ready for submission.')